# `GaussianCGO.project`, step by step

## The objects

* Any **operator** $O$ is the sum of some **Majorana strings**: $O = \sum_A c_A\,\gamma_A$,
  where $c_A$ are complex numbers and $\gamma_A = \prod_{a \in A}\gamma_a$ is a product of
  **Majoranas** $\gamma_a$ (Hermitian, $\gamma_a^2 = 1$,
  $\gamma_a\gamma_b = -\gamma_b\gamma_a$ for $a \neq b$).
* The **patch** is the physical patch $L$ that we keep; the modes outside it are traced out.
  The **covariance block** $\Gamma_L$ is the submatrix of the state's covariance on the
  patch's Majoranas.
* The **Williamson decomposition** of that block writes it as
  $\Gamma_L = R^{T}\,\mathrm{diag}(\lambda_k J)\,R$ with
  $J = \begin{bmatrix}0&-1\\1&0\end{bmatrix}$ and $R$ real orthogonal, so every
  $0 \leq \lambda_k \leq 1$.  Each $\lambda_k$ belongs to one **normal mode** of the patch,
  whose Majoranas are $\tilde\gamma_b = \sum_a R_{ba}\gamma_a$.
* After Williamson decomposition, each Fermion modes are now independent, the density matrix of the whole Fock space is the tensor product of the density matrix of each mode:
$$\rho_L = \bigotimes_k \rho_k,\qquad
  \rho_k = (1-p_k)\,|\tilde 0\rangle_k\langle\tilde 0|
        + p_k\,|\tilde 1\rangle_k\langle\tilde 1|,
  \qquad p_k = \frac{1-\lambda_k}{2}.$$
  - $\lambda_k = 1$, i.e. $p_k = 0$: only $|\tilde 0\rangle_k$ carries weight and $\rho_k$ has
  rank one -- the mode is **pure**.
  - $0 < \lambda_k < 1$, i.e. $0 < p_k \le \frac12$: both weights are positive and $\rho_k$ has
  rank two -- the mode is **mixed**.
* Hence, $\mathrm{Im}\,\rho_L$ is the tensor product of images, $\mathrm{Im}\,\rho_k$ is spanned by $|\tilde 0\rangle_k$ if $k$-th mode is pure and by
$|\tilde 0\rangle_k$ and $|\tilde 1\rangle_k$ if the $k$-th mode is mixed:

$$V_L = \mathrm{Im}\,\rho_L = \Big(\bigotimes_{k\ \text{mixed}}
  \mathrm{span}\{|\tilde 0\rangle_k, |\tilde 1\rangle_k\}\Big)
  \;\otimes\; \Big(\bigotimes_{k\ \text{pure}} |\tilde 0\rangle_k\Big),
  \qquad \dim V_L = 2^{\#\text{mixed}},$$

* The natural (canonical basis) in $V_L$ is thus, labeled by states on the mixed modes: $\ket{\tilde{n}_1\dots\tilde{n}_{\# \text{mixed}}} = \ket{\tilde{n}_1}\otimes\dots\otimes\ket{\tilde{n}_{\# \text{mixed}}}$, with $\tilde{n}_k=0,1$.

One detail of the labelling.  The decomposition returns $\lambda_k = |2p_k^{\text{raw}} - 1|$ of
the raw occupation of a mode, so a mode that is more than half occupied is written with the
opposite orientation and always appears above with $p_k \le \frac12$.  Neither $\rho_L$ nor
$V_L$ depends on that choice; only the naming of $|\tilde 0\rangle_k$ and $|\tilde 1\rangle_k$
does.

## What has to be computed

The matrix of $O$ on $V_L$.  Since $|j\rangle \in V_L$ and $\langle i| = \langle i|P_V$ with
$P_V$ the projector on $V_L$, restricting $O$ to $V_L$ does not change its elements, so

$$M_{ij} = (P_V O P_V)_{ij} = \langle i|\,O\,|j\rangle ,
\qquad i, j = 0, \dots, 2^{\#\text{mixed}} - 1,$$

with $|j\rangle$ the basis state of the canonical basis above whose occupations
$(\tilde n_1, \dots, \tilde n_{\#\text{mixed}})$ are the binary digits of $j$, most significant
digit first, and with every pure mode in $|\tilde 0\rangle$.

## How to get one element

1. Expand every Majorana in the normal modes, $\gamma_a = \sum_b R_{ba}\tilde\gamma_b$.  A term
   of $O$ then becomes a sum of products of normal-mode Majoranas, one choice of $b$ for each
   label of the term.
2. The occupation of a mode is itself a Majorana bilinear,
   $n_k = (1 + i\tilde\gamma_{2k}\tilde\gamma_{2k+1})/2$.  The Majorana $\tilde\gamma_{2k}$
   anticommutes with that bilinear, so it flips $n_k$; and it anticommutes with the Majorana
   that builds each occupied mode below $k$, so moving it into place costs one $-1$ per such
   mode: the factor $(-1)^{\sum_{j<k}n_j}$. (From the arranging in Fock space) The odd member of the pair,
   $\tilde\gamma_{2k+1}$, carries an extra factor $i(-1)^{n_k}$.  So one Majorana maps a basis
   state to a sign times exactly one other basis state.
3. Applying that to $|j\rangle$ gives a sum of such terms; the coefficient of $|i\rangle$ in
   that sum is $M_{ij}$.
4. The pure modes are not an index: a piece of the sum in which a pure mode ends up occupied
   is not in $V_L$ and is dropped.  That drop is the projector $P_V$; a mixed mode may end in
   either of its two states.
5. Add the pieces with their coefficients $c_A$.

The rest of the notebook is how the code carries out these five steps.


In [ ]:
import bisect
import itertools
from functools import reduce

import torch

from FreeFermion.GfPEPS import Bond, Graph, GfPEPS
from FreeFermion.linalg import RandPureCov, hamiltonian
from GaussianCGO import GaussianCGO

torch.manual_seed(0)

PHYSICAL, ENVIRONMENT = 8, 8

whole = RandPureCov(PHYSICAL + ENVIRONMENT)  # [physical | environment]
cgo = GaussianCGO(whole, list(range(PHYSICAL)))

print(f'patch   Majoranas {cgo.internal_index}')
print(f'modes   {cgo.modes}   pure {cgo.pure}   mixed {cgo.mixed}')
print(f'size    {cgo.size} = 2**{len(cgo.mixed)}, the mixed Fock space')
print(f'subsets {2 ** len(cgo.pure)} = 2**{len(cgo.pure)}, the pure register')
print(f'fock    {2 ** cgo.modes} = 2**{cgo.modes} if the pure modes were kept too')


ValueError: too many values to unpack (expected 2, got 3)

## The code

The function `project` in `CGO/GaussianCGO.py`, in the order it runs.  The class labels
Majoranas by integers; its constructor takes the Majorana labels of the patch as
`internal`, keeps the complement as `external_index` and traces it out, and the block it hands
to the Williamson -- the one every operator below is written in -- is the patch block
`self.internal_block`.  From the physics above the code takes
three things: the Williamson
rotation as `self.rotation` (the matrix $R$), the sorted lists `self.mixed` and `self.pure` of
the two kinds of modes, and the projector as "every pure mode in $|\tilde 0\rangle$".

Two conventions come from `GaussianCGO.__init__` and `FreeFermion/linalg.py` and are used
throughout below.  The Williamson eigenvalues are the eigenvalues of $i\Gamma$ taken with a
positive sign, so a mode counts as pure exactly when $\lambda_k > 1 - \texttt{tolerance}$, and
the decomposition orients each mode the emptier way: the code keeps that oriented vacuum, so a
pure mode is taken to be empty whatever its raw orientation was (the class says "a pure mode is
fixed to its ground state").

Skeleton of the function, with the section that takes each piece:

```python
matrix = torch.zeros(...)                     # 1  the result, filled one term at a time
identity = torch.eye(...)                     # 1  the state the operator is applied to
row = torch.arange(...)                       # 2  the row index: one mixed state per entry
parity = plus.clone()
for bit in range(len(self.mixed)):            # 3  the parity sign, tabulated over the rows
    parity = torch.where(...)
for coefficient, labels in term:              # 6  one term c_A gamma_A at a time
    if any(label not in self.position ...):   # 6  every label must belong to the patch
        raise ValueError(...)
    state = {0: identity}                     # 6  the running state, per pure subset
    for label in labels:                      # 5  the Majoranas of the term, left to right
        column = self.rotation[:, ...]        # 4  the coefficients R_ba of that Majorana
        next_state = {}                       # 5  the sum over the normal modes b
        for mode, odd in itertools.product(...):   # 4  one normal mode, both of its Majoranas
            element = column[2 * mode + odd]  # 4  R_ba
            if not element: continue          # 4  a zero coefficient contributes nothing
            ... below, sign, pure_mask ...    # 4  the sign of the modes below
            if mode in self.mixed: ...        # 4  the flip of a mixed mode
            else: ...                         # 4  the flip of a pure mode
            for subset, amplitude in state.items():   # 5  every running amplitude at once
                ... next_state[target] = ...  # 5  accumulate the pieces
        state = next_state                    # 5  this Majorana has been applied
    if 0 in state:                            # 6  the projection: keep the pure vacuum
        matrix = matrix + coefficient * ...   # 6  add the term
return matrix
```

## 1. The result, and the state it is applied to

```python
matrix = torch.zeros((self.size, self.size), dtype=COMPLEX, device=self.device)
identity = torch.eye(self.size, dtype=COMPLEX, device=self.device)
```

`matrix` is the `size x size` result, and `identity` is the state that the operator is applied
to.  Applying an operator to the identity is what computes all elements at once: column $j$ of
the identity is $|j\rangle$, so column $j$ of the product is $O|j\rangle$, whose entries are
$\langle i|O|j\rangle$.  Nothing else has to write a basis state in later.

The operator is applied to the state one term at a time, and the running state is carried as a
dictionary from the occupations of the **pure** modes to the amplitude on the mixed ones --
because a pure mode has to end where it started, in $|\tilde 0\rangle$, and while a term is
being applied it may be occupied:

```python
state = {0: identity}          # inside the loop over the terms, see section 6
```

* the **key** is `subset`, one integer whose bits are the occupations of the **pure** modes:
  bit `p` of `subset` is 1 exactly when mode `pure[p]` is occupied.  Reading the bits of one
  integer as occupations is what makes the key a single number; such an integer is a
  **bitmask**.  The bits are **little-endian**, `pure[0]` owning bit 0, the least significant
  one -- the opposite of the row index of the next section.
* the **value** is the amplitude on the mixed modes, a matrix of shape `(size, size)`.

So the dictionary is the operator's action, kept separately for every value of the pure
occupations; the cell below lists the keys it can take.


In [ ]:
print(f'{"subset":>6}  excited pure modes')
for subset in range(2 ** len(cgo.pure)):
    excited = [mode for position, mode in enumerate(cgo.pure) if (subset >> position) & 1]
    print(f'{subset:>6}  {excited}')


## 2. The row index

```python
row = torch.arange(self.size, device=self.device)
```

`row` is the index of every entry of an amplitude: one mixed basis state per entry, in the
order the amplitude stores them.  Row `r` is the basis state in which the mixed mode at
position `s` of `self.mixed` is occupied exactly when bit `len(mixed)-1-s` of `r` is 1, so the
first mixed mode owns the most significant bit and $|r\rangle$ reads like
$|\tilde n_0\tilde n_1\cdots\rangle$ from the left.  This register is **big-endian**, the
opposite convention to the subset key of the previous section, and it is the $i$ of $M_{ij}$
in the opening.

The whole array is built once, because the flip of section 4 is a bit operation on it: `row`
appears again in `row ^ mask` and `row & below`.


In [ ]:
print(f'{"row":>4} {"binary":>7}  occupied mixed modes')
for index in range(cgo.size):
    bits = format(index, f'0{len(cgo.mixed)}b')
    occupied = [mode for position, mode in enumerate(cgo.mixed) if bits[position] == '1']
    print(f'{index:>4} {bits:>7}  {occupied}')


## 3. The parity sign of a mode, tabulated over the rows

```python
plus = torch.ones(self.size, dtype=COMPLEX, device=self.device)
parity = plus.clone()
for bit in range(len(self.mixed)):
    parity = torch.where(((row >> bit) & 1) == 0, parity, -parity)
```

Acting with a normal-mode Majorana on a basis state flips one occupation and
multiplies by $(-1)^{\sum_{j<k}n_j}$.  The occupation is the bilinear
$n_k = (1 + i\tilde\gamma_{2k}\tilde\gamma_{2k+1})/2$, with which $\tilde\gamma_{2k}$
anticommutes, and the Majorana of each occupied mode below $k$ anticommutes with it as well,
one $-1$ each.

The loop above builds that parity for *every* set of modes at once.  Each pass flips the sign
of `parity[r]` for the rows whose bit `bit` is set, so after the loop
`parity[r] = (-1)**popcount(r)`, where the **popcount** is the number of 1 bits of an integer
(`int.bit_count` in Python).  A set of modes below a mode $k$ is a contiguous set of bits, and
its mask is computed in the next section, so the sign of any mode is then one lookup in this
table -- the code never multiplies a matrix for it.


In [ ]:
row = torch.arange(cgo.size, device=cgo.device)
plus = torch.ones(cgo.size, dtype=torch.complex128, device=cgo.device)
parity = plus.clone()
for bit in range(len(cgo.mixed)):
    parity = torch.where(((row >> bit) & 1) == 0, parity, -parity)

print(f'{"row":>4} {"binary":>7} {"popcount":>8} {"parity":>7}')
for index in range(cgo.size):
    bits = format(index, f'0{len(cgo.mixed)}b')
    print(f'{index:>4} {bits:>7} {bits.count("1"):>8} {int(parity[index].real):>7}')

mode = cgo.mixed[1]
mixed_count = bisect.bisect_left(cgo.mixed, mode)
below = ((1 << mixed_count) - 1) << (len(cgo.mixed) - mixed_count)
sign = parity[row & below]
print(f'\nmode {mode}: below = {below:0{len(cgo.mixed)}b}, '
      f'Z string = {[int(value.real) for value in sign]}')


## 4. One normal mode of one Majorana

A Majorana is expanded in the normal modes, $\gamma_a = \sum_b R_{ba}\tilde\gamma_b$, and a
normal mode $b$ is a pair of Majoranas, `mode = b // 2` and `odd = b % 2`.  The first three
lines of the loop over a term take that expansion apart for the label at hand:

```python
column = self.rotation[:, self.position[label]].tolist()
...
for mode, odd in itertools.product(range(self.modes), (False, True)):
    element = column[2 * mode + odd]
    if not element:
        continue
```

`column` is the column $R_{\cdot a}$ of the Williamson rotation for the Majorana $\gamma_a$,
so `element` is one coefficient $R_{ba}$: the `(mode, odd)` pair picks the normal mode `mode`
and, within it, the even ($\tilde\gamma_{2k}$, `odd = False`) or the odd ($\tilde\gamma_{2k+1}$, `odd = True`) member of the pair.  The
`continue` drops a coefficient that vanishes, which is the only shortcut taken here.

For that normal mode the loop next computes the two masks and the sign:

```python
    mixed_count = bisect.bisect_left(self.mixed, mode)
    pure_count = bisect.bisect_left(self.pure, mode)
    below = ((1 << mixed_count) - 1) << (len(self.mixed) - mixed_count)
    sign = parity[row & below]
    pure_mask = (1 << pure_count) - 1
```

`mixed_count` and `pure_count` are how many mixed and pure modes lie below `mode` -- the lists
`self.mixed` and `self.pure` are ascending, so a bisection counts them.  `below` is those mixed
modes as the top bits of the row index, and `sign` is the parity sign of section 3 for them, as a
$\pm 1$ per row; `pure_mask` is the same mask for the pure modes, as the low bits of the subset.

The mode the operator acts on is then moved, and this is the **flip**:

```python
    if mode in self.mixed:
        mask = 1 << (len(self.mixed) - 1 - self.mixed.index(mode))
        flip, subset_flip = row ^ mask, 0
        if odd:
            sign = sign * torch.where((row & mask) == 0, 1j * plus, -1j * plus)
    else:
        flip, subset_flip = slice(None), 1 << self.pure.index(mode)
        if odd:
            sign = sign * 1j
            pure_mask |= subset_flip
```

A Majorana moves exactly one occupation, so a basis state is mapped to the state with that one
bit changed, and the matrix element of that state has to be read from the partner.  For a
**mixed** mode the occupation is a bit of the row index: `flip = row ^ mask` sends every row to
its partner, the subset is untouched (`subset_flip = 0`), and the odd Majorana adds the factor
$i(-1)^{n_k}$, read off that bit of the row.  For a **pure** mode the occupation is a bit of
the subset: `subset_flip` changes it, and `flip = slice(None)` leaves the rows alone -- a basic
slice in torch is a **view**, sharing the memory of the original, so the rows pass through
without being copied.  There the odd Majorana's factor is the constant $i$ times
$(-1)^{n_k}$, and $n_k$ is the same subset bit that `subset_flip` moves, which is why that bit
is `or`-ed into `pure_mask` and picked up by the parity test of the next section.

The cell below prints `below`, `pure_mask` and `flip` for every mode of the patch, which is
this whole section in one table.


In [ ]:
WIDTH = len(cgo.mixed)
print(f'{"b":>2} {"mode":>4} {"mixed":>5} {"odd":>5} {"below":>6} {"pure_mask":>9}  flip')
for mode in range(cgo.modes):
    mixed_count = bisect.bisect_left(cgo.mixed, mode)
    pure_count = bisect.bisect_left(cgo.pure, mode)
    below = ((1 << mixed_count) - 1) << (len(cgo.mixed) - mixed_count)
    pure_mask = (1 << pure_count) - 1
    mixed = mode in cgo.mixed
    if mixed:
        mask = 1 << (len(cgo.mixed) - 1 - cgo.mixed.index(mode))
        flip = (row ^ mask).tolist()
    else:
        flip = 'row'
    for odd in (False, True):
        land = pure_mask | (1 << cgo.pure.index(mode)) if (odd and not mixed) else pure_mask
        print(f'{2 * mode + odd:>2} {mode:>4} {str(mixed):>5} {str(odd):>5} '
              f'{below:>0{WIDTH}b} {land:>0{len(cgo.pure)}b}  {flip}')


## 5. Summing a Majorana over its normal modes

```python
next_state: dict[int, Tensor] = {}
...
        for subset, amplitude in state.items():
            weight = -sign if (subset & pure_mask).bit_count() % 2 else sign
            piece = element * (weight[:, None] * amplitude)[flip]
            target = subset ^ subset_flip
            previous = next_state.get(target)
            next_state[target] = piece if previous is None else previous + piece
    state = next_state
```

One Majorana is the sum over the normal modes $b$, so the pieces are accumulated into the new
dictionary `next_state` rather than written over `state`:

* `weight` is the parity sign of section 4, and the extra factor $(-1)^{n_k}$ of an odd acting on a
  pure mode: `(subset & pure_mask).bit_count() % 2` is the parity of the pure modes below the
  one being moved, plus that mode itself when it is odd.
* `(weight[:, None] * amplitude)[flip]`: `weight` is a vector over the rows, so multiplying
  scales every column by it, and indexing with `flip` is the row operation of section 4 --
  entry $i$ of the result comes from entry $\mathrm{flip}(i)$ of the scaled amplitude.
* `piece` multiplies by the coefficient $R_{ba}$ of that normal mode.
* `target` is the pure subset the piece belongs to, and two normal modes that reach the same
  subset add their pieces there (`previous + piece`).

Writing into `state` instead is not possible: the loop is reading `state`, and for a mixed
mode `target == subset`, so an entry would be read after it had been overwritten.  At the end
of the mode loop, `state = next_state` is "this Majorana has been applied".

The cell below runs this loop with printing added, one line per normal mode, and checks that
it reproduces `project` itself.


In [ ]:
LABEL = cgo.internal_index[2]
column = cgo.rotation[:, cgo.position[LABEL]].tolist()
state = {0: torch.eye(cgo.size, dtype=torch.complex128, device=cgo.device)}

next_state = {}
print(f'{"b":>2} {"mode":>4} {"odd":>5} {"element":>9} {"weight":>26}  targets')
for mode, odd in itertools.product(range(cgo.modes), (False, True)):
    element = column[2 * mode + odd]
    if not element:
        continue
    mixed_count = bisect.bisect_left(cgo.mixed, mode)
    pure_count = bisect.bisect_left(cgo.pure, mode)
    below = ((1 << mixed_count) - 1) << (len(cgo.mixed) - mixed_count)
    sign = parity[row & below]
    pure_mask = (1 << pure_count) - 1
    if mode in cgo.mixed:
        mask = 1 << (len(cgo.mixed) - 1 - cgo.mixed.index(mode))
        flip, subset_flip = row ^ mask, 0
        if odd:
            sign = sign * torch.where((row & mask) == 0, 1j * plus, -1j * plus)
    else:
        flip, subset_flip = slice(None), 1 << cgo.pure.index(mode)
        if odd:
            sign = sign * 1j
            pure_mask |= subset_flip
    targets = []
    for subset, amplitude in state.items():
        weight = -sign if (subset & pure_mask).bit_count() % 2 else sign
        piece = element * (weight[:, None] * amplitude)[flip]
        target = subset ^ subset_flip
        previous = next_state.get(target)
        next_state[target] = piece if previous is None else previous + piece
        targets.append(target)
    marks = ' '.join(f'{value.real:+.0f}{value.imag:+.0f}i' for value in weight.tolist())
    landing = 'row' if isinstance(flip, slice) else flip.tolist()
    print(f'{2 * mode + odd:>2} {mode:>4} {str(odd):>5} {element:>9.5f} '
          f'{marks:>26}  {targets}  flip {landing}')

print(f'\nsubsets after one Majorana: {sorted(next_state)}   (state had {sorted(state)})')
print('subset 0 alone, against project(*[(1.0, [LABEL])]): '
      f'{float((next_state[0] - cgo.project(*[(1.0, [LABEL])])).abs().max()):.1e}')


## 6. One term of the string, and the projection

```python
for coefficient, labels in term:
    if any(label not in self.position for label in labels):
        raise ValueError(...)
    state = {0: identity}
    for label in labels:
        ...                                     # sections 4 and 5, once per label
    if 0 in state:
        matrix = matrix + coefficient * state[0]
return matrix
```

`term` is the list of $(c_A, A)$ pairs of $O = \sum_A c_A\gamma_A$, so the outer loop takes
one term at a time.  The check raises if a label is not a mode of the patch, since the
covariance block of a patch says nothing about a Majorana outside it.

Each term starts from the vacuum of the pure modes with the mixed modes untouched,
`{0: identity}`, and its labels are applied one after another, in the order they are written.
The dictionary then holds one amplitude per pure subset, and only the key 0 is read: the terms
whose pure modes do not end in $|\tilde 0\rangle$ have left $V_L$, and dropping them is the
projection $P_V$.  What is added to `matrix` is $c_A$ times that block, i.e.
$c_A P_V\gamma_A P_V$, restricted to $V_L$.

The cell below builds one string this way and compares it with `project` itself.


In [ ]:
OBSERVABLE = [(1.0, [0, 1, 2, 3])]
matrix = cgo.project(*OBSERVABLE)

print('cgo.project(*[(1.0, [0, 1, 2, 3])])')
for line in matrix.cpu():
    print('  '.join(f'{value.real:>8.4f}{value.imag:+.4f}i' for value in line))
print(f'largest |M - M^dagger|: {float((matrix - matrix.conj().T).abs().max()):.1e}')


## 7. The same matrix, built the long way

The check below builds the same matrix without any of the bit bookkeeping, so that the two
constructions can differ only if one of them is wrong.  Every mode is one tensor factor of the
full $2^{\#\text{modes}}$-dimensional space, so the Majoranas can be written down explicitly as
products of Pauli matrices,
$\gamma_{2k} = Z^{\otimes k}\otimes X\otimes I$ and
$\gamma_{2k+1} = Z^{\otimes k}\otimes Y\otimes I$, with mode `k` the $(k+1)$-th factor from the
left, so that the $Z$ factors sit on the modes below it and the identity on the modes above.
Expanding them in the normal modes and projecting the pure ones onto $|\tilde 0\rangle$ gives

$$P_V = \prod_{k\ \text{pure}} \frac{I - i\tilde\gamma_{2k}\tilde\gamma_{2k+1}}{2},$$

which is $V_L = \mathrm{Im}\,\rho_L$ inside that full space: the mixed modes are kept whole and
the pure ones sit in vacuum.  These products are Hermitian -- equal to their own conjugate
transpose, so their eigenvalues are real -- which is what `eigvalsh` computes.  The two
matrices then have the same spectrum, differing only by the order of the mixed basis.  One
detail: `project` applies the labels of a term left to right, so what it builds is the product
in reverse label order, and the reference below multiplies them in that same order.


In [ ]:
ident = torch.eye(2, dtype=torch.complex128, device=cgo.device)
X = torch.tensor([[0, 1], [1, 0]], dtype=torch.complex128, device=cgo.device)
Y = torch.tensor([[0, -1j], [1j, 0]], dtype=torch.complex128, device=cgo.device)
Z = torch.tensor([[1, 0], [0, -1]], dtype=torch.complex128, device=cgo.device)

gamma = [reduce(torch.kron, [Z] * (a // 2) + [X if a % 2 == 0 else Y]
                + [ident] * (cgo.modes - a // 2 - 1))
         for a in range(2 * cgo.modes)]
normal = [sum(float(cgo.rotation[b, a]) * gamma[a] for a in range(2 * cgo.modes))
          for b in range(2 * cgo.modes)]

dimension = 2 ** cgo.modes
identity = torch.eye(dimension, dtype=torch.complex128, device=cgo.device)
projector = identity
for k in cgo.pure:
    projector = projector @ (identity - 1j * normal[2 * k] @ normal[2 * k + 1]) / 2
values, vectors = torch.linalg.eigh(projector)
basis = vectors[:, values.real > 0.5]
print(f'dimension of V_L from the projector: {basis.shape[1]}, size {cgo.size}')

flat = hamiltonian(cgo.internal_block).cpu()
labels = cgo.internal_index
strings = {
    'i gamma_0 gamma_1': [(1j, [0, 1])],
    'gamma_0..gamma_3': [(1.0, [0, 1, 2, 3])],
    'gamma_0 gamma_1 gamma_4 gamma_5': [(1.0, [0, 1, 4, 5])],
    'H_L': [(0.5j * float(flat[a, b]), [labels[a], labels[b]])
            for a in range(len(labels)) for b in range(a + 1, len(labels))],
}
print(f'\n{"string":<34}{"max |eigenvalue difference|":>28}')
for name, term in strings.items():
    operator = torch.zeros((dimension, dimension), dtype=torch.complex128, device=cgo.device)
    for coefficient, term_labels in term:
        product = identity
        for label in reversed(term_labels):
            product = product @ gamma[cgo.position[label]]
        operator = operator + coefficient * product
    reference = torch.linalg.eigvalsh(basis.conj().T @ operator @ basis)
    ours = torch.linalg.eigvalsh(cgo.project(*term))
    print(f'{name:<34}{float((reference - ours).abs().max()):>28.1e}')


## 8. What to read in the code

Every piece above is a piece of `project` in `CGO/GaussianCGO.py`; this table is the map from
the mathematics to the code.

| in `project` | section | meaning |
|---|---|---|
| `matrix`, `identity` | 1 | the result; the state the operator is applied to, one basis state per column |
| `state = {0: identity}` | 1, 6 | the running state, one amplitude per occupied pure subset |
| `row = torch.arange(self.size)` | 2 | the mixed register, big-endian |
| `parity[...]` | 3 | $(-1)^{\text{popcount}}$, the parity sign of any set of modes in one lookup |
| `column`, `element` | 4 | the column $R_{\cdot a}$ and one coefficient $R_{ba}$ |
| `bisect.bisect_left` | 4 | how many mixed (pure) modes are below `mode` |
| `below`, `pure_mask`, `sign` | 4 | those modes as masks, and the parity sign on every row |
| `flip, subset_flip` | 4 | the row flip of a mixed mode, the subset flip of a pure one |
| `weight`, `piece`, `target` | 5 | the parity sign on the pure half; one term of the expansion; the subset it lands on |
| `next_state` | 5 | the sum over the normal modes, and why it is a second dictionary |
| `if 0 in state` | 6 | the projection $P_V$ |
| `matrix = matrix + coefficient * state[0]` | 6 | the term $c_A P_V \gamma_A P_V$ |
